# Uniformize Data Formats
The study incorporates six datasets. 

The first is a proprietary dataset on a [systematic review on pancreatic surgery.](10.1016/j.surg.2021.04.023)

The dataset was provided by EVIglance Inc. but cannot be shared publicly.


The remaining five datasets are from the open-source [SYNERGY repository on GitHub](https://github.com/asreview/synergy-dataset).

## Loading the data
Locate the files housing the original data:

In [1]:
from os import listdir
from os.path import isfile, join

# where the original data is located
directory = '../../../data/datasets/00_original'

files = [f for f in listdir(directory) if isfile(join(directory, f))] 
subjects = [file.split('_raw.')[0] for file in files] # without extension

## Data Dictionary
To easily access all datasets programmatically, group them within one data dictionary.

Note how the initial data on pancreatic surgery was a ``.tsv``-file which the code accounts for with a ``synergy`` attribute.

In [2]:
import polars as pl

datasets = {
    subjects[count]: {
        'dataframe': pl.read_csv(
            join(directory, file), 
            separator='\t' if file.endswith('.tsv') else ',', 
            encoding='utf-8'),
        'synergy': True if file.endswith('.csv') else False
    } for count, file in enumerate(files)
}

## Inspection
At this point, the original study compared the structure of the proprietary and open-source datasets.
Since the dataset on pancreatic surgery cannot be published, we simply inspect one example from the SYNERGY dataset.

Notice two things here:

1. There are no titles or abstracts yet. These have to be retrieved from [OpenAlex](https://openalex.org/)
2. The ``label_included`` column is numerically encoded and ambiguous.

In [3]:
datasets['animal_depression']['dataframe'].head()

doi,pmid,openalex_id,label_included,method
str,str,str,i64,str
"""https://doi.org/10.1042/bj1300…","""https://pubmed.ncbi.nlm.nih.go…","""https://openalex.org/W24010252…",0,"""id_retrieval_pmid"""
null,"""https://pubmed.ncbi.nlm.nih.go…","""https://openalex.org/W24105122…",0,"""id_retrieval_pmid"""
null,null,"""https://openalex.org/W24180790…",0,"""search_title"""
"""https://doi.org/10.1111/ejn.12…","""https://pubmed.ncbi.nlm.nih.go…","""https://openalex.org/W20173882…",1,"""id_retrieval_pmid"""
"""https://doi.org/10.1097/000032…","""https://pubmed.ncbi.nlm.nih.go…","""https://openalex.org/W19957205…",0,"""id_retrieval_pmid"""


## Uniformize

Next, the study uniformized the two types of datasets.

While we can not demonstrate uniformization, note how the preliminary new format provides a boolean ``include`` label column, and one column each for title and abstract:


| include 	| title 	| abstract 	| doi 	| literatureid 	| openalex_id 	|
|---------	|-------	|----------	|-----	|------	|-------------	|
| bool    	| str   	| str      	| str 	| str  	| str         	|

First, define a function to bring the dataframes to the new format:

In [5]:
import re

def uniformize(dataframe: pl.DataFrame, synergy: bool) -> pl.DataFrame:
    
    # There are different column names across the datasets
    label_column = 'label_included' if synergy else 'State'
    doi = 'doi' if synergy else 'Doi'
    id_column = 'pmid' if synergy else 'LiteratureId'

    # Logic to map the labels to a boolean value
    exclude_label = 0 if synergy else 3
    mapping = lambda x: False if x == exclude_label else True

    # Map the labels to boolean values
    labels = dataframe[label_column].to_list()
    includes = [mapping(label) for label in labels]

    # Different literature id formats within one column comprise:
    # web of science (WOS:), cochrane central (CN-), pubmed (), and hand-signed (HS-)
    id_formats = r'(WOS:|CN-|HS-)*([A-Z]|\d)+$'

    # extract the identifier from the url within the literature_id column
    def extract_id(id):
        if id is not None and id != "":
            match = re.search(id_formats, str(id))
            return match.group() if match else None
        else:
            return None

    literature_urls = dataframe[id_column].to_list()
    literature_ids = [extract_id(id) for id in literature_urls]


    return pl.DataFrame(
        {
            'include': includes,
            'title': None if synergy else dataframe['Title'].to_list(),
            'abstract': None if synergy else dataframe['Abstract'].to_list(),
            'doi': dataframe[doi].to_list(),
            'literature_id': literature_ids,
            'openalex_id': dataframe['openalex_id'].to_list() if synergy else [None] * dataframe.height,
        }
    )


Apply the function to uniformize all dataframes and save them to a new dictionary:

In [6]:
uniform_datasets = {key: uniformize(
    value['dataframe'], value['synergy']) for key, value in datasets.items()}

Verify that the dataframes are now in the correct format:

In [7]:
uniform_datasets['adhd'].head()

include,title,abstract,doi,literature_id,openalex_id
bool,null,null,str,str,str
false,null,null,"""https://doi.org/10.1007/bf0301…","""10051933""","""https://openalex.org/W20826139…"
false,null,null,"""https://doi.org/10.1056/nejm19…","""10053177""","""https://openalex.org/W23126093…"
false,null,null,"""https://doi.org/10.1037/0021-8…","""10066996""","""https://openalex.org/W20229048…"
false,null,null,"""https://doi.org/10.1097/000005…","""10072008""","""https://openalex.org/W20210973…"
false,null,null,"""https://doi.org/10.1056/nejm19…","""10072410""","""https://openalex.org/W42392839…"


### Export

In [ ]:
directory_to_save = '../../../data/datasets/01_uniform'

[dataframe.write_csv(f'{directory_to_save}/{subject}_uniform.csv') 
    for subject, dataframe in uniform_datasets.items() ]